# PRÁCTICA 3 SEMANA 2

**Se recomienda ejecutar los códigos de este jupyer notebook en la misma carpeta donde reside dicho documento, debido a la dependencia de ciertos documentos para el correcto uso del mismo.**

En primer lugar, vamos a importar las librerías necesarias y vamos a importar los csv necesarios.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
import scipy.stats as stats
from sklearn.metrics import r2_score
from sklearn.dummy import DummyClassifier
from scipy.stats import zscore
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegressionCV
from statsmodels.api import Logit

In [3]:
explotacion = pd.read_csv("fuga_clientes_empresa_telefonica_explotacion.csv")
construccion = pd.read_csv("fuga_clientes_empresa_telefonica_construccion.csv")

En la semana 2 nos vamos a centrar principalmente en hacer un análisis más detallado del modelo de regresión logística. En este caso, vamos a usar la librería statsmodels. Primero de todo, vamos a traer el preprocesamiento de la semana 1 y vamos a crear el modelo de regresión.

In [4]:
# Utilizamos str.extract para eliminar los strings de la id para que no nos de problema
explotacion["Customer ID"] = explotacion["Customer ID"].str.extract(r"(\d+)")
explotacion["Customer ID"] = explotacion["Customer ID"].astype(int)
construccion["Customer ID"] = construccion["Customer ID"].str.extract(r"(\d+)")
construccion["Customer ID"] = construccion["Customer ID"].astype(int)

# Quitamos el target
explotacion_mod = explotacion.drop(columns=["Churn Status"])
construccion_mod = construccion.drop(columns=["Churn Status"])

# Añadimos un identificador temporal para poder separar después del get_dummies
explotacion_mod['_dataset'] = 'explotacion'
construccion_mod['_dataset'] = 'construccion'

# Concatenamos los dataframes
combined = pd.concat([construccion_mod, explotacion_mod], axis=0, ignore_index=True)

# Aplicar get_dummies a todo el dataset concatenado
# Así se crean las mismas columnas para ambos datasets
combined = pd.get_dummies(combined, columns=["Network type subscription in Month 1"], drop_first=True)
combined = pd.get_dummies(combined, columns=["Network type subscription in Month 2"], drop_first=True)
combined = pd.get_dummies(combined, columns=["Most Loved Competitor network in Month 1"], drop_first=True)
combined = pd.get_dummies(combined, columns=["Most Loved Competitor network in Month 2"], drop_first=True)

# Separamos de nuevo en construccion y explotacion
construccion_mod = combined[combined['_dataset'] == 'construccion'].copy()
explotacion_mod = combined[combined['_dataset'] == 'explotacion'].copy()

# Eliminamos el identificador temporal
construccion_mod = construccion_mod.drop(columns=['_dataset'])
explotacion_mod = explotacion_mod.drop(columns=['_dataset'])

# Calculamos matrices de covarianza
cov_construccion = construccion_mod.cov()
cov_explotacion = explotacion_mod.cov()

# Calculamos los rangos
rango_construccion = np.linalg.matrix_rank(cov_construccion)
rango_explotacion = np.linalg.matrix_rank(cov_explotacion)

# Definimos cols y calculamos su rango
cols = list(construccion_mod.columns)
rank = np.linalg.matrix_rank(construccion_mod[cols].cov())
# Generamos un bucle while para que lo haga para cada columna
i = 0
while i < len(cols):
    col = cols[i]
    # Creamos una copia de las columnas sin la actual
    cols2 = cols.copy()
    cols2.remove(col)
    # Calculamos el rango sin esa columna
    construccion_mod_2 = construccion_mod[cols2]
    rank2 = np.linalg.matrix_rank(construccion_mod_2.cov())
    # Si el rango no cambia, eliminamos definitivamente la columna
    if rank == rank2:
        cols = cols2
        i = 0
    else:
        i += 1
        
# Ahora eliminamos las variables dependientes redefiniendo los 2 dataframes
construccion_mod = construccion_mod[cols]
explotacion_mod = explotacion_mod[cols]

# Eliminamos outliers
z_construccion = construccion_mod.apply(zscore)
z_explotacion = explotacion_mod.apply(zscore)
explotacion_mod = explotacion_mod[(z_explotacion.abs() < 3).all(axis=1)]
construccion_mod = construccion_mod[(z_construccion.abs() < 3).all(axis=1)]

# Eliminamos las columnas constantes para que no de problemas a la hora de entrenar modelos
columnas_constantes = construccion_mod.columns[construccion_mod.std() == 0].tolist()
if len(columnas_constantes) > 0:
    for col in columnas_constantes:
        construccion_mod = construccion_mod.drop(columns=columnas_constantes)
        explotacion_mod = explotacion_mod.drop(columns=columnas_constantes)

In [5]:
explotacion_mod

,Customer ID,Customer tenure in month,Total Spend in Months 1 and 2 of 2017,Total SMS Spend,Total Data Spend,Total Unique Calls,Total Onnet spend,Total Offnet spend,Total Call centre complaint calls,Network type subscription in Month 2_3G,Most Loved Competitor network in Month 2_Uxaa
1112,42,44.33,60.5036,0.03,32.50,14,36,1876,2,False,False
1113,65,3.73,418.4264,0.00,95.00,43,672,3713,2,False,True
1114,81,11.70,893.5700,0.60,0.00,405,0,563,3,True,True
1115,98,49.73,1061.8480,14.64,0.00,430,6642,713,2,False,True
1117,106,32.70,1208.6004,1.75,2.50,883,18345,7434,2,False,True
...,...,...,...,...,...,...,...,...,...,...,...
1393,1972,19.90,763.9200,88.87,31.25,82,7155,21202,1,False,False
1395,1978,42.27,898.3100,137.65,5.00,84,26304,27511,1,False,True
1396,1979,3.83,298.5780,0.00,5.27,14,847,228,4,True,True
1397,1982,139.67,967.0904,25.37,115.00,36,1235,9209,1,False,True


In [ ]:
construccion_mod

#### PRIMERA PARTE
Con los dataframes preprocesados vamos a pasar a creaer el modelo de regresión logística. A su vez, tendremos que compararlo con el modelo creado con sklearn. Por eso vamos a traer el modelo del anterior jupyter y vamos a comparar sus R2_score. Además, vamos a sacar la información del modelo de regresión logística para tener un análisis más avanzado.

Primero vamos a definir el train y el test para entrenar los modelos correspondientes

In [ ]:
# Definimos el eje y como el target y el eje x como el dataframe ya preprocesado
y_construccion = construccion.loc[construccion_mod.index, "Churn Status"]   
x_construccion = construccion_mod
x_explotacion = explotacion_mod

# IMPORTANTE: El dataset de construcción se divide en train y test
# El dataset de explotación se usa solo para predicción final
x_train, x_test, y_train, y_test = train_test_split(
    x_construccion, # Eje x del modelo
    y_construccion, # Eje y del modelo
    test_size=0.2, # Tomamos el 20% del dataset como test
    random_state=42,
    stratify=y_construccion)

# Tenemos que estandarizar x_train y x_test para el modelo
x_train_scaled = x_train
x_test_scaled = x_test

for col in x_train.columns:
    media = x_train[col].mean()
    desviacion = x_train[col].std()
    x_train_scaled[col] = (x_train[col] - media) / desviacion
    x_test_scaled[col] = (x_test[col] - media) / desviacion

Ahora creamos los modelos y calculamos sus R2_score y vemos los coeficientes.

In [ ]:
# Creamos ambos modelos
LR_model_Sk = LogisticRegression()
LR_model_Sm = Logit(y_train,x_train_scaled)

# Entrenamos los modelos
LR_model_Sk.fit(x_train_scaled, y_train)
Sm_model = LR_model_Sm.fit()

# R2_score
score_train_lr_Sk = LR_model_Sk.score(x_train_scaled, y_train)
score_test_lr_Sk = LR_model_Sk.score(x_test_scaled, y_test)
score_train_lr_Sm = Sm_model.prsquared

print(f"R2_Score de train LogisticRegression_Sk: {score_train_lr_Sk}")
print(f"R2_Score de test LogisticRegression_Sk: {score_test_lr_Sk}")
print(f"R2_Score de train LogisticRegression_Sm: {score_train_lr_Sm}")

In [ ]:
# Calculamos los parametros y los comparamos para ver si hay mucha diferencia.
parametros_Sk = np.array(LR_model_Sk.coef_.tolist())
parametros_Sm = np.array(Sm_model.params.tolist())
# Los reconvertimos para visualizarlos correctamente
parametros_Sk = LR_model_Sk.coef_.flatten()
parametros_Sm = Sm_model.params.values
# Vamos a crear un dataframe para visualizarlo mejor.
df_comparacion = pd.DataFrame({
    "StatsModels": parametros_Sm,
    "ScikitLearn": parametros_Sk,
    "Diferencia": parametros_Sm - parametros_Sk
}, index=Sm_model.params.index)

df_comparacion

De los resultados que hemos obtenido vamos a fijarnos primeramente en los R2_score. Podemos ver como el modelo generado con sckit tiene una precisión mayor además de no tener sobreajuste. La sintaxis que se usa para sckit además de ser más lógica y simple en mi opinión, nos da la opción de ver si hay sobreajuste en el modelo únicamente con los score de train y test. Sin embargo, con statsmodels no podemos verlo a simple vista. 

El modelo de statsmodels nos da un modelo con un R2 de un 20%, un modelo bastante malo comparado con el que ha sido generado por sckit con un 70%. Comparando los coeficientes de ambos modelos vemos que no hay una diferencia significativa. Una diferencia de $p < 0.05$. Sin embargo, si nos fijamos en los coeficientes de los modelos vemos que hay varios $p > 0.5$. 

Si tuvieramos que elegir un modelo por los coeficientes elegiría sckit. Al comparar ambos modelos sckit tiene coeficientes mucho menores y más cercanos a 0 que statsmodels.

In [ ]:
Sm_model.summary()

Viendo el summary de nuestro modelo de statsmodels podemos ver todos los datos que nos interesan, como los coeficientes, los p-values y los intervalos de confianza. Si queremos hallar las variables que no aportan información nos tenemos que fijar en $P > |z|$ que es la fila de los p-values. Aquellos que $p < 0.05$ serán significativos para predecir o para nuestro modelo. Si nos fijamos detenidamente vemos que las variables significativas son:

"Total Spend in Months 1 and 2 of 2017", "Total SMS Spend", "Total Onnet spend", "Total Offnet spend", "Most Loved Competitor network in Month 2_Uxaa"

#### SEGUNDA PARTE:

Una vez hemos creado nuestro modelo de regresión logística con statsmodels y lo hemos analizado interiormente vamos a seleccionar las variables clave. En la parte 1 hemos identificado las variables significativas, ahora vamos a implementar un algoritmo que elimine las no significativas para tener un modelo todavía mejor y ver sis upera a los antiguos modelos.

Para ello vamos a inspirarnos en la práctica 2.

In [ ]:
# Generamos las columnas y sus condiciones
alpha = 0.05
cols = list(x_train_scaled.columns)
print("Columnas antes del algoritmo",len(cols))
# Generamos el algoritmo
while True:
    # Ajusta el modelo según se va actualizando
    modelo_temp = Logit(y_train, x_train_scaled[cols]).fit()
    # Guardamos los p-values
    p_values = np.array(modelo_temp.pvalues.to_list())
    max_pval = p_values.max()
    # Break para salir del modelo
    if max_pval <= alpha:
        break
    # Quita la variable con mayor p-value
    worst_var = modelo_temp.pvalues.idxmax()
    cols.remove(worst_var)

print("Columnas después del algoritmo",len(cols))
print(cols)

# Ahora calculamos los scores y comparamos
print(Sm_model.prsquared)
print(modelo_temp.prsquared)

Hemos implementado el modelo correctamente y se han quedado las variables que habíamos recalcado en el apartado 1. Ahora tenemos este modelo con las variables más significativas. En total se han eliminado unas 6 variables. A la hora de calcular los scores vemos que ha disminuido en 1 punto porcentual. Vemos que no es mucha diferencia pero aun así, seleccionando las variables más relevantes el modelo no mejora si no que empeora.

#### TERCERA PARTE:

Dado que el mejor modelo obtenido hasta ahora es el de sckit-learn, vamos a volver a implementarlo y vamos a compararlo detenidamente con el modelo dummy que también hemos creado previamente.

## PEDRO BARROS BOBADILLA LEE
El mejor modelo hasta el momento es el logisticRegression de sckit, está en este jupyter. El de dummy está en la primera semana